# Data audit / EDA

Roadmap stage 3. This notebook is self-contained: it downloads the
dataset itself (same as `01_dataset_download_colab.ipynb`), so it can be
run on its own in a fresh Colab session without depending on another
notebook having run first.

## Step 3.1 — Verify dataset integrity

Goal: confirm all 200 class folders are present, and that every image file
actually opens correctly (not just a sample — with ~116K files this is
fast enough to check all of them and get a definitive answer).

### Setup: download the dataset (from Google Drive, authenticated)

Uses Colab's built-in Google auth + the Drive API to download by file ID,
as the file's own owner — not the public "anyone with the link" path.
This avoids Google Drive's anti-abuse download quota on shared links
(which we hit during testing: "Too many users have viewed or downloaded
this file recently"). You'll get a one-time login/consent popup.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import io
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

FILE_ID = "1Mi0IleRucNmwnQ4g_ZEWOyBFFv4mO9ba"
ZIP_PATH = "/content/traffic_sign_dataset.zip"

drive_service = build('drive', 'v3')
request = drive_service.files().get_media(fileId=FILE_ID)

with io.FileIO(ZIP_PATH, 'wb') as fh:
    downloader = MediaIoBaseDownload(fh, request, chunksize=200 * 1024 * 1024)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        print(f"Download progress: {int(status.progress() * 100)}%")

print("Download complete.")

In [ ]:
import os

EXPECTED_SIZE = 14_945_416_206  # bytes, from the local audit

assert os.path.exists(ZIP_PATH), (
    "Download failed: no file was written at all. Check the gdown output above for errors."
)
actual_size = os.path.getsize(ZIP_PATH)
print(f"Downloaded file size: {actual_size:,} bytes ({actual_size / 1e9:.2f} GB)")
print(f"Expected size: {EXPECTED_SIZE:,} bytes ({EXPECTED_SIZE / 1e9:.2f} GB)")

assert actual_size > 1_000_000_000, (
    "Downloaded file is far too small to be the real dataset (likely Google Drive's "
    "'can't scan this file for viruses' warning page instead of the actual zip). "
    "Check that the Drive file's sharing is set to 'Anyone with the link', then re-run "
    "the download cell above."
)
print("Size check passed — looks like the real file.")

In [ ]:
!rm -rf /content/dataset
!mkdir -p /content/dataset
!unzip -q "$ZIP_PATH" -d /content/dataset
!rm "$ZIP_PATH"

from pathlib import Path
data_path = Path('/content/dataset/Data')
assert data_path.is_dir(), (
    "Unzip did not produce /content/dataset/Data as expected — the zip may be "
    "corrupt or incomplete. Re-run the download cell and check its output for errors."
)
n_classes = len(list(data_path.iterdir()))
print(f"Unzip done, zip file removed. Found {n_classes} folders under Data/.")

### 3.1a — Confirm all 200 class folders are present

In [ ]:
from pathlib import Path

data_dir = Path('/content/dataset/Data')
class_folders = sorted(data_dir.iterdir(), key=lambda p: int(p.name))

found = set(p.name for p in class_folders)
expected = set(str(i) for i in range(200))

print(f"Found {len(class_folders)} class folders")
print("Missing classes:", sorted(expected - found, key=int) or "None")
print("Unexpected extra folders:", sorted(found - expected) or "None")

### 3.1b — Verify every image actually opens (full check, not a sample)

In [ ]:
from PIL import Image
from tqdm import tqdm

corrupt_files = []
total_checked = 0

for class_folder in tqdm(class_folders, desc="Checking classes"):
    for img_path in class_folder.iterdir():
        total_checked += 1
        try:
            with Image.open(img_path) as img:
                img.verify()
        except Exception as e:
            corrupt_files.append((str(img_path), str(e)))

print(f"\nChecked {total_checked} images")
print(f"Corrupt/unreadable: {len(corrupt_files)}")
for path, err in corrupt_files[:20]:
    print(" ", path, "->", err)

### Result

If class folders = 200 with none missing, and corrupt/unreadable = 0 (or a
short, specific list), step 3.1 is done. Any corrupt files found here get
excluded before training later. Next: step 3.2, class distribution.

## Step 3.2 — Class distribution

Goal: quantify how many images each of the 200 classes has, and confirm
the imbalance already spotted locally (248 to 1,845 images per class).

In [ ]:
import numpy as np

class_counts = {cf.name: len(list(cf.iterdir())) for cf in class_folders}
counts = np.array(list(class_counts.values()))

min_class = min(class_counts, key=class_counts.get)
max_class = max(class_counts, key=class_counts.get)

print(f"Number of classes: {len(counts)}")
print(f"Total images: {counts.sum()}")
print(f"Min images in a class: {counts.min()} (class {min_class})")
print(f"Max images in a class: {counts.max()} (class {max_class})")
print(f"Mean: {counts.mean():.1f}, Median: {np.median(counts):.1f}")
print(f"Imbalance ratio (max/min): {counts.max() / counts.min():.2f}x")

In [ ]:
import matplotlib.pyplot as plt

# Bar chart in class-ID order
sorted_by_id = sorted(class_counts.items(), key=lambda x: int(x[0]))
plt.figure(figsize=(20, 5))
plt.bar([x[0] for x in sorted_by_id], [x[1] for x in sorted_by_id])
plt.xlabel('Class ID')
plt.ylabel('Number of images')
plt.title('Images per class, in class-ID order')
plt.xticks([])
plt.tight_layout()
plt.show()

# Same data sorted by count, so the imbalance is easy to see as a shape
sorted_by_count = sorted(class_counts.items(), key=lambda x: x[1])
plt.figure(figsize=(20, 5))
plt.bar(range(len(sorted_by_count)), [x[1] for x in sorted_by_count], color='steelblue')
plt.axhline(counts.mean(), color='red', linestyle='--', label=f'Mean ({counts.mean():.0f})')
plt.xlabel('Classes, sorted from fewest to most images')
plt.ylabel('Number of images')
plt.title('Class distribution sorted (shows the imbalance shape)')
plt.legend()
plt.tight_layout()
plt.show()

### Result

This confirms (or corrects) the imbalance we spotted earlier. The sorted
chart's shape matters more than exact numbers here: a gentle slope means
mild imbalance, a steep drop-off for a handful of classes means a few
classes are seriously underrepresented and will need special handling
(class weighting and/or extra augmentation) before training. Next: step
3.3, image properties (size, aspect ratio, color mode).

## Step 3.3 — Image property audit

Goal: check resolution range, aspect ratios, color mode, and file size
across the dataset. This tells us how to resize/normalize images
consistently before training.

In [ ]:
import pandas as pd

records = []
for class_folder in tqdm(class_folders, desc="Reading image properties"):
    class_id = class_folder.name
    for img_path in class_folder.iterdir():
        with Image.open(img_path) as img:
            width, height = img.size
            mode = img.mode
        file_size = img_path.stat().st_size
        records.append({
            'class_id': class_id,
            'width': width,
            'height': height,
            'mode': mode,
            'file_size_bytes': file_size,
        })

props_df = pd.DataFrame(records)
props_df['aspect_ratio'] = props_df['width'] / props_df['height']
print(f"Total images analyzed: {len(props_df)}")
props_df.head()

In [ ]:
print("--- Width (px) ---")
print(props_df['width'].describe())
print("\n--- Height (px) ---")
print(props_df['height'].describe())
print("\n--- Aspect ratio (width / height) ---")
print(props_df['aspect_ratio'].describe())
print("\n--- File size (bytes) ---")
print(props_df['file_size_bytes'].describe())
print("\n--- Color mode counts ---")
print(props_df['mode'].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(props_df['width'], bins=50, color='steelblue')
axes[0, 0].set_title('Width distribution')
axes[0, 0].set_xlabel('Width (px)')

axes[0, 1].hist(props_df['height'], bins=50, color='steelblue')
axes[0, 1].set_title('Height distribution')
axes[0, 1].set_xlabel('Height (px)')

axes[1, 0].hist(props_df['aspect_ratio'], bins=50, color='steelblue')
axes[1, 0].set_title('Aspect ratio (width / height) distribution')
axes[1, 0].axvline(1.0, color='red', linestyle='--', label='Square (1:1)')
axes[1, 0].legend()

mode_counts = props_df['mode'].value_counts()
axes[1, 1].bar(mode_counts.index.astype(str), mode_counts.values, color='steelblue')
axes[1, 1].set_title('Color mode counts (log scale)')
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.show()

### Result

Key decisions this feeds into stage 4 (preprocessing):
- **Target resize size**: pick a fixed size (commonly small, e.g. 32x32 or
  64x64 for sign-classification CNNs) based on the resolution range above —
  no point training at a resolution higher than what most source images
  actually have.
- **Aspect ratio**: if most images are close to square (ratio near 1.0),
  resizing to a square target won't distort signs much. If there's a wide
  spread, we may want padding instead of a plain stretch-resize.
- **Color mode**: if it's overwhelmingly RGB with only a few exceptions,
  we'll just convert those exceptions to RGB rather than redesigning the
  pipeline around them.

Next: step 3.4, visual sample audit (actually look at example images).

## Step 3.4 — Visual sample audit

Goal: actually look at example images from a handful of classes — the
cheapest possible sanity check that a folder's images genuinely match one
consistent sign, not a labeling mistake. Class names aren't mapped yet
(that's step 3.5), so classes are shown by ID for now.

In [ ]:
import random

random.seed(42)  # reproducible sample, same images every run

n_sample_classes = 8
images_per_class = 4

sample_class_ids = random.sample([cf.name for cf in class_folders], n_sample_classes)

fig, axes = plt.subplots(
    n_sample_classes, images_per_class,
    figsize=(images_per_class * 2.2, n_sample_classes * 2.2)
)

for row, class_id in enumerate(sample_class_ids):
    class_path = data_dir / class_id
    img_files = list(class_path.iterdir())
    chosen = random.sample(img_files, min(images_per_class, len(img_files)))
    for col in range(images_per_class):
        ax = axes[row, col]
        if col < len(chosen):
            with Image.open(chosen[col]) as img:
                ax.imshow(img.convert('RGB'))
            ax.set_title(f"class {class_id}", fontsize=9)
        ax.axis('off')

plt.suptitle('Random sample: 4 images from each of 8 random classes', y=1.01)
plt.tight_layout()
plt.show()

### Result

Check each row: do all 4 images in a row genuinely look like the *same*
sign (just different angle/lighting/distance), and does it look like a
real, distinct traffic sign (not a blank, a mistake, or an unrelated
photo)? If something looks wrong, note the class ID and we'll dig into
that specific folder. If it all looks consistent, step 3.4 is done. Next:
step 3.5, mapping class IDs to human-readable names.